In [1]:
import pandas as pd
import numpy as np

In [ ]:
TRAIN_PATH = '../train.csv'
TEST_PATH = '../test.csv'


train = pd.read_csv(TRAIN_PATH, parse_dates=['time'])
test = pd.read_csv(TEST_PATH, parse_dates=['time'])

In [3]:
group_cols = ['station', 'layer']

train = train.sort_values(
    group_cols + ['time']
).reset_index(drop=True)

test = test.sort_values(
    group_cols + ['time']
).reset_index(drop=True)

In [ ]:
import numpy as np
import pandas as pd


def make_features(df):

    df = df.copy()

    group_cols = ['station', 'layer']


    # =========================================================
    # 0. 정렬
    # =========================================================

    df = df.sort_values(
        group_cols + ['time']
    ).reset_index(drop=True)


    # =========================================================
    # 1. 시간 및 계절성
    # =========================================================

    df['month'] = df['time'].dt.month
    df['dayofyear'] = df['time'].dt.dayofyear
    df['hour'] = df['time'].dt.hour

    df['month_sin'] = np.sin(
        2 * np.pi * df['month'] / 12
    )

    df['month_cos'] = np.cos(
        2 * np.pi * df['month'] / 12
    )


    # =========================================================
    # 2. 단기 온도 차분
    # =========================================================

    for lag in [1, 2, 3]:

        df[f'temp_diff_{lag}'] = (
            df.groupby(group_cols)['temp']
              .diff(lag)
        )

        df[f'abs_temp_diff_{lag}'] = (
            df[f'temp_diff_{lag}'].abs()
        )


    # =========================================================
    # 3. Rolling 통계
    # =========================================================

    windows = [
        3, 6, 12, 24, 48,
        72, 144, 288, 576
    ]

    for w in windows:

        grouped = (
            df.groupby(group_cols)['temp']
              .rolling(
                  window=w,
                  min_periods=max(3, w // 2)
              )
        )

        rolling_mean = (
            grouped.mean()
            .reset_index(
                level=[0, 1],
                drop=True
            )
        )

        rolling_std = (
            grouped.std()
            .reset_index(
                level=[0, 1],
                drop=True
            )
        )

        rolling_min = (
            grouped.min()
            .reset_index(
                level=[0, 1],
                drop=True
            )
        )

        rolling_max = (
            grouped.max()
            .reset_index(
                level=[0, 1],
                drop=True
            )
        )

        rolling_median = (
            grouped.median()
            .reset_index(
                level=[0, 1],
                drop=True
            )
        )

        df[f'temp_mean_{w}'] = rolling_mean
        df[f'temp_std_{w}'] = rolling_std

        df[f'temp_range_{w}'] = (
            rolling_max - rolling_min
        )

        df[f'temp_residual_{w}'] = (
            df['temp'] - rolling_mean
        )

        df[f'temp_dev_median_{w}'] = (
            df['temp'] - rolling_median
        )

        df[f'temp_zscore_{w}'] = (
            (df['temp'] - rolling_mean)
            / (rolling_std + 1e-6)
        )


    # =========================================================
    # 4. 장기 변화량
    # =========================================================

    for lag in [6, 12, 24, 48, 72]:

        df[f'temp_change_{lag}'] = (
            df['temp']
            - df.groupby(group_cols)['temp'].shift(lag)
        )

        df[f'abs_temp_change_{lag}'] = (
            df[f'temp_change_{lag}'].abs()
        )


    # =========================================================
    # 5. Flatline
    # =========================================================

    df['temp_diff_abs'] = (
        df.groupby(group_cols)['temp']
          .diff()
          .abs()
    )

    df['is_same_temp'] = (
        df['temp_diff_abs'] < 1e-6
    ).astype(int)


    def consecutive_same(x):

        result = np.zeros(
            len(x),
            dtype=int
        )

        count = 0

        for i, value in enumerate(x):

            if value:
                count += 1
            else:
                count = 0

            result[i] = count

        return pd.Series(
            result,
            index=x.index
        )


    df['flatline_length'] = (
        df.groupby(group_cols)['is_same_temp']
          .transform(consecutive_same)
    )


    # =========================================================
    # 6. 보조 변수
    # =========================================================

    if 'psal' in df.columns:

        df['psal_diff'] = (
            df.groupby(group_cols)['psal']
              .diff()
        )

    if 'depth' in df.columns:

        df['depth_diff'] = (
            df.groupby(group_cols)['depth']
              .diff()
        )


    # =========================================================
    # 7. Rolling slope
    # =========================================================

    def rolling_slope(arr):

        if len(arr) < 3:
            return np.nan

        valid = ~np.isnan(arr)

        if valid.sum() < 3:
            return np.nan

        idx = np.arange(len(arr))

        return np.polyfit(
            idx[valid],
            arr[valid],
            1
        )[0]


    for w in [72, 144, 288, 519]:

        df[f'temp_slope_{w}'] = (
            df.groupby(group_cols)['temp']
              .transform(
                  lambda x:
                  x.rolling(
                      w,
                      min_periods=w // 3
                  )
                  .apply(
                      rolling_slope,
                      raw=True
                  )
              )
        )


    # =========================================================
    # 8. Sign consistency
    # =========================================================

    for w in [72, 144, 288]:

        sign = np.sign(
            df[f'temp_residual_{w}']
        )

        df[f'temp_sign_consistency_{w}'] = (

            sign
            .groupby(
                [df['station'], df['layer']]
            )
            .transform(
                lambda x:
                x.rolling(
                    w,
                    min_periods=w // 2
                )
                .mean()
                .abs()
            )
        )


    # =========================================================
    # 9. ★ 과거 기반 Past-only feature
    # =========================================================
    #
    # 현재값을 제외하고 직전 관측값들만 사용
    #
    # 현재 temp
    #     ↓
    # 과거 window와 비교
    #
    # offset / drift 탐지 목적
    # =========================================================

    past_temp = (
        df.groupby(group_cols)['temp']
          .shift(1)
    )


    past_grouped = (
        past_temp
        .groupby(
            [df['station'], df['layer']]
        )
    )


    for w in [12, 24, 48, 72, 144, 288, 576]:

        # -----------------------------------------------------
        # 과거 평균
        # -----------------------------------------------------

        past_mean = (
            past_grouped
            .rolling(
                w,
                min_periods=max(3, w // 2)
            )
            .mean()
            .reset_index(
                level=[0, 1],
                drop=True
            )
        )


        # -----------------------------------------------------
        # 과거 표준편차
        # -----------------------------------------------------

        past_std = (
            past_grouped
            .rolling(
                w,
                min_periods=max(3, w // 2)
            )
            .std()
            .reset_index(
                level=[0, 1],
                drop=True
            )
        )


        # -----------------------------------------------------
        # 과거 중앙값
        # -----------------------------------------------------

        past_median = (
            past_grouped
            .rolling(
                w,
                min_periods=max(3, w // 2)
            )
            .median()
            .reset_index(
                level=[0, 1],
                drop=True
            )
        )


        # -----------------------------------------------------
        # feature 저장
        # -----------------------------------------------------

        df[f'past_mean_{w}'] = past_mean

        df[f'past_std_{w}'] = past_std

        df[f'past_median_{w}'] = past_median


        # -----------------------------------------------------
        # 현재값 - 과거 평균
        # -----------------------------------------------------

        df[f'past_residual_{w}'] = (
            df['temp'] - past_mean
        )


        # -----------------------------------------------------
        # 현재값 - 과거 중앙값
        # -----------------------------------------------------

        df[f'past_dev_median_{w}'] = (
            df['temp'] - past_median
        )


        # -----------------------------------------------------
        # 과거 기준 z-score
        # -----------------------------------------------------

        df[f'past_zscore_{w}'] = (
            (df['temp'] - past_mean)
            / (past_std + 1e-6)
        )


        # -----------------------------------------------------
        # 절대 편차
        # -----------------------------------------------------

        df[f'past_abs_residual_{w}'] = (
            df[f'past_residual_{w}'].abs()
        )


        df[f'past_abs_dev_median_{w}'] = (
            df[f'past_dev_median_{w}'].abs()
        )


    # =========================================================
    # 10. ★ 단기 slope vs 장기 slope
    # =========================================================

    df['slope_diff_72_288'] = (
        df['temp_slope_72']
        - df['temp_slope_288']
    )

    df['slope_diff_144_288'] = (
        df['temp_slope_144']
        - df['temp_slope_288']
    )

    df['abs_slope_72'] = (
        df['temp_slope_72'].abs()
    )

    df['abs_slope_144'] = (
        df['temp_slope_144'].abs()
    )

    df['abs_slope_288'] = (
        df['temp_slope_288'].abs()
    )


    
    return df

In [27]:
train['dataset'] = 'train'
test['dataset'] = 'test'

all_data = pd.concat(
    [train, test],
    ignore_index=True
)

all_data = all_data.sort_values(
    ['station', 'layer', 'time']
).reset_index(drop=True)

all_feat = make_features(all_data)

train_feat = all_feat[
    all_feat['dataset'] == 'train'
].copy()

test_feat = all_feat[
    all_feat['dataset'] == 'test'
].copy()

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_23036\4213463443.py:381: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'past_residual_{w}'] = (
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_23036\4213463443.py:390: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'past_dev_median_{w}'] = (
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_23036\4213463443.py:399: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfor

MemoryError: Unable to allocate 1.04 GiB for an array with shape (179, 776706) and data type float64

feature 확인

In [ ]:
print("Train shape:", train_feat.shape)
print("Test shape :", test_feat.shape)

print("\n===== Feature columns =====")
for i, col in enumerate(train_feat.columns):
    print(i, col)

Train shape: (776706, 158)
Test shape : (169011, 158)

===== Feature columns =====
0 station
1 year
2 layer
3 time
4 temp
5 psal
6 depth
7 label
8 anomaly_type
9 dataset
10 month
11 dayofyear
12 hour
13 month_sin
14 month_cos
15 temp_diff_1
16 abs_temp_diff_1
17 temp_diff_2
18 abs_temp_diff_2
19 temp_diff_3
20 abs_temp_diff_3
21 temp_mean_3
22 temp_std_3
23 temp_range_3
24 temp_residual_3
25 temp_dev_median_3
26 temp_zscore_3
27 temp_mean_6
28 temp_std_6
29 temp_range_6
30 temp_residual_6
31 temp_dev_median_6
32 temp_zscore_6
33 temp_mean_12
34 temp_std_12
35 temp_range_12
36 temp_residual_12
37 temp_dev_median_12
38 temp_zscore_12
39 temp_mean_24
40 temp_std_24
41 temp_range_24
42 temp_residual_24
43 temp_dev_median_24
44 temp_zscore_24
45 temp_mean_48
46 temp_std_48
47 temp_range_48
48 temp_residual_48
49 temp_dev_median_48
50 temp_zscore_48
51 temp_mean_72
52 temp_std_72
53 temp_range_72
54 temp_residual_72
55 temp_dev_median_72
56 temp_zscore_72
57 temp_mean_144
58 temp_std_144
59 

In [ ]:
original_cols = [
    'station',
    'year',
    'layer',
    'time',
    'temp',
    'psal',
    'depth',
    'label',
    'dataset'
]

feature_cols = [
    col for col in train_feat.columns
    if col not in original_cols
]

print("\nFeature 개수:", len(feature_cols))
print(feature_cols)


Feature 개수: 149
['anomaly_type', 'month', 'dayofyear', 'hour', 'month_sin', 'month_cos', 'temp_diff_1', 'abs_temp_diff_1', 'temp_diff_2', 'abs_temp_diff_2', 'temp_diff_3', 'abs_temp_diff_3', 'temp_mean_3', 'temp_std_3', 'temp_range_3', 'temp_residual_3', 'temp_dev_median_3', 'temp_zscore_3', 'temp_mean_6', 'temp_std_6', 'temp_range_6', 'temp_residual_6', 'temp_dev_median_6', 'temp_zscore_6', 'temp_mean_12', 'temp_std_12', 'temp_range_12', 'temp_residual_12', 'temp_dev_median_12', 'temp_zscore_12', 'temp_mean_24', 'temp_std_24', 'temp_range_24', 'temp_residual_24', 'temp_dev_median_24', 'temp_zscore_24', 'temp_mean_48', 'temp_std_48', 'temp_range_48', 'temp_residual_48', 'temp_dev_median_48', 'temp_zscore_48', 'temp_mean_72', 'temp_std_72', 'temp_range_72', 'temp_residual_72', 'temp_dev_median_72', 'temp_zscore_72', 'temp_mean_144', 'temp_std_144', 'temp_range_144', 'temp_residual_144', 'temp_dev_median_144', 'temp_zscore_144', 'temp_mean_288', 'temp_std_288', 'temp_range_288', 'temp_r

In [ ]:
#train에서 안나온 값
missing = pd.DataFrame({
    'missing_count': train_feat[feature_cols].isna().sum(),
    'missing_ratio': train_feat[feature_cols].isna().mean() * 100
})

print(
    missing.sort_values(
        'missing_ratio',
        ascending=False
    )
)

                         missing_count  missing_ratio
anomaly_type                    744580      95.863815
psal_diff                        18512       2.383399
past_dev_median_576               4608       0.593275
past_abs_residual_576             4608       0.593275
past_abs_dev_median_576           4608       0.593275
...                                ...            ...
month_sin                            0       0.000000
month                                0       0.000000
dayofyear                            0       0.000000
flatline_length                      0       0.000000
is_same_temp                         0       0.000000

[149 rows x 2 columns]


In [ ]:
#테스트에서 안나온 값
missing_test = pd.DataFrame({
    'missing_count': test_feat[feature_cols].isna().sum(),
    'missing_ratio': test_feat[feature_cols].isna().mean() * 100
})

print("\n===== TEST missing =====")
print(
    missing_test.sort_values(
        'missing_ratio',
        ascending=False
    )
)


===== TEST missing =====
                    missing_count  missing_ratio
anomaly_type               169011     100.000000
depth_diff                  16391       9.698185
psal_diff                    1064       0.629545
month                           0       0.000000
dayofyear                       0       0.000000
...                           ...            ...
slope_diff_72_288               0       0.000000
slope_diff_144_288              0       0.000000
abs_slope_72                    0       0.000000
abs_slope_144                   0       0.000000
abs_slope_288                   0       0.000000

[149 rows x 2 columns]


In [ ]:
#2026년에서 큰 결측비율
drop_features = [
    'depth_diff'
]

In [ ]:
print("\n피처 값 기본 통계")

print(
    train_feat[feature_cols]
    .describe()
    .T
)


피처 값 기본 통계
                       count        mean        std           min         25%  \
month               776706.0    7.024086   2.987689  1.000000e+00    5.000000   
dayofyear           776706.0  198.356311  90.746427  1.000000e+00  134.000000   
hour                776706.0   11.501895   6.928239  0.000000e+00    5.000000   
month_sin           776706.0   -0.192717   0.677725 -1.000000e+00   -0.866025   
month_cos           776706.0   -0.135460   0.696564 -1.000000e+00   -0.866025   
...                      ...         ...        ...           ...         ...   
slope_diff_72_288   775186.0   -0.000009   0.021572 -4.004304e-01   -0.004971   
slope_diff_144_288  775186.0   -0.000004   0.008123 -1.450256e-01   -0.001677   
abs_slope_72        776338.0    0.011437   0.018753  2.975722e-18    0.001799   
abs_slope_144       775954.0    0.004724   0.008664  1.267361e-17    0.000786   
abs_slope_288       775186.0    0.002708   0.005143  3.139706e-09    0.000487   

               

flatline_length의 max 값이 너무 큰데?
ㄴ그럴 수 있대
온도가 직전 시점과 완전 똑같은 경우 
1%미만에 불ㅜ함 

282x10분
47시간동안 완전히 고정된 부분 



In [ ]:
#station 별 분포

for station in train_feat['station'].unique():

    sub = train_feat[
        train_feat['station'] == station
    ]

    print(f"\n===== {station} =====")

    print(
        sub
        .groupby('label')[check_features]
        .median()
        .T
    )


===== G-ORS =====
label                 0.0       1.0
abs_temp_diff_1  0.027600  0.112100
abs_temp_diff_2  0.039400  0.121000
abs_temp_diff_3  0.050300  0.148700
temp_std_3       0.029004  0.204274
temp_std_6       0.046754  0.385677
temp_std_12      0.073758  0.584300
temp_std_24      0.111883  0.761255
temp_std_48      0.162728  0.872670
temp_zscore_3    0.021601  0.000000
temp_zscore_6    0.022135  0.000000
temp_zscore_12  -0.001416  0.000000
temp_zscore_24   0.033536  0.000000
temp_zscore_48   0.040956  0.000000
flatline_length  0.000000  0.000000

===== I-ORS =====
label                 0.0       1.0
abs_temp_diff_1  0.037200  0.019300
abs_temp_diff_2  0.058000  0.032800
abs_temp_diff_3  0.076200  0.045600
temp_std_3       0.040566  0.021144
temp_std_6       0.068392  0.038214
temp_std_12      0.113017  0.066971
temp_std_24      0.183921  0.125188
temp_std_48      0.271455  0.234728
temp_zscore_3   -0.010631  0.000000
temp_zscore_6    0.009338  0.000000
temp_zscore_12   0.028981 

In [ ]:
for station in train_feat['station'].unique():

    print(f"\n\n========== {station} ==========")

    result = (
        train_feat[
            train_feat['station'] == station
        ]
        .groupby(['layer', 'label'])['abs_temp_diff_1']
        .median()
        .unstack()
    )

    print(result)



========== G-ORS ==========
label     0.0     1.0
layer                
1      0.0276  0.1121


========== I-ORS ==========
label      0.0      1.0
layer                  
1      0.02500  0.02800
2      0.03910  0.03135
3      0.05590  0.03220
4      0.10045  0.00015
5      0.02900  0.03800
6      0.05850  0.00000
7      0.02270  0.01180


========== S-ORS ==========
label     0.0      1.0
layer                 
1      0.0202  0.05660
2      0.0441  0.11530
3      0.0673  0.66765
4      0.1488  0.08580
5      0.0461  0.02440
6      0.1419  0.35610
7      0.0129  0.11435
8      0.0080  0.00350


feature 하나만 사용했을때 이상치를 얼마나 잘 구분하는지 

auc :feature 값이 클수록 이상치인지, 작을수록 이상치인지 고려해서 평가

best f1: feature 하나만 가지고 threhold를 최적으로 잡았을 때 f1
threhold 란? (한계점)


separation : label 1/ 0 이상의 평균 차이를 표준편차로 정규화한 값

In [ ]:
# ============================================================
# Feature별 단독 분리력 비교 전체적으로. no group
# AUC / Best F1 / Separation
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    precision_recall_curve
)


# ============================================================
# 1. Feature 목록
# ============================================================

drop_cols = [
    'station',
    'year',
    'layer',
    'time',
    'label',
    'anomaly_type',
    'dataset',
    'depth_diff'
]

feature_cols = [
    col for col in train_feat.columns
    if col not in drop_cols
]


# ============================================================
# 2. Feature별 평가
# ============================================================

results = []

y = train_feat['label'].astype(int)


for feature in feature_cols:

    data = train_feat[
        [feature, 'label']
    ].dropna()

    if data[feature].nunique() <= 1:
        continue

    x = data[feature].values
    y_true = data['label'].astype(int).values

    # --------------------------------------------
    # AUC
    # --------------------------------------------

    try:
        auc_raw = roc_auc_score(
            y_true,
            x
        )
    except:
        continue

    # 값이 작을수록 anomaly인 feature도 있으므로
    # 방향을 뒤집어서 항상 AUC >= 0.5가 되도록 함
    if auc_raw >= 0.5:

        auc = auc_raw
        direction = 'higher'

        score = x

    else:

        auc = 1 - auc_raw
        direction = 'lower'

        score = -x


    # --------------------------------------------
    # Best F1
    # --------------------------------------------

    precision, recall, thresholds = precision_recall_curve(
        y_true,
        score
    )

    if len(thresholds) > 0:

        f1_scores = (
            2 * precision[:-1] * recall[:-1] /
            (
                precision[:-1] +
                recall[:-1] +
                1e-12
            )
        )

        best_idx = np.argmax(f1_scores)

        best_f1 = f1_scores[best_idx]
        best_threshold = thresholds[best_idx]

        pred = (
            score >= best_threshold
        ).astype(int)

        best_precision = precision_score(
            y_true,
            pred,
            zero_division=0
        )

        best_recall = recall_score(
            y_true,
            pred,
            zero_division=0
        )

    else:

        best_f1 = np.nan
        best_threshold = np.nan
        best_precision = np.nan
        best_recall = np.nan


    # --------------------------------------------
    # Separation
    # Cohen's d
    # --------------------------------------------

    normal = x[y_true == 0]
    anomaly = x[y_true == 1]

    mean_normal = np.mean(normal)
    mean_anomaly = np.mean(anomaly)

    std_normal = np.std(normal, ddof=1)
    std_anomaly = np.std(anomaly, ddof=1)

    pooled_std = np.sqrt(
        (
            (len(normal) - 1) * std_normal**2 +
            (len(anomaly) - 1) * std_anomaly**2
        )
        /
        (
            len(normal) +
            len(anomaly) -
            2
        )
    )

    separation = (
        abs(mean_anomaly - mean_normal)
        / (pooled_std + 1e-12)
    )


    # --------------------------------------------
    # 저장
    # --------------------------------------------

    results.append({
        'feature': feature,
        'AUC': auc,
        'Best_F1': best_f1,
        'Precision': best_precision,
        'Recall': best_recall,
        'Separation': separation,
        'direction': direction,
        'threshold_score': best_threshold,
        'normal_mean': mean_normal,
        'anomaly_mean': mean_anomaly,
        'N': len(data)
    })


# ============================================================
# 3. 결과 DataFrame
# ============================================================

feature_results = pd.DataFrame(results)


# Best F1 기준
feature_results = feature_results.sort_values(
    'Best_F1',
    ascending=False
).reset_index(drop=True)


print(
    feature_results[
        [
            'feature',
            'AUC',
            'Best_F1',
            'Precision',
            'Recall',
            'Separation',
            'direction'
        ]
    ].to_string(index=False)
)

                  feature      AUC  Best_F1  Precision   Recall  Separation direction
          flatline_length 0.596601 0.322291   0.994527 0.192305    1.839170    higher
             is_same_temp 0.596407 0.313868   0.806884 0.194827    2.102130    higher
  past_abs_dev_median_576 0.722993 0.274382   0.278097 0.270765    1.556566    higher
    past_abs_residual_576 0.729210 0.264524   0.278380 0.251982    1.497055    higher
  past_abs_dev_median_288 0.683445 0.236463   0.193923 0.302911    1.211227    higher
    past_abs_residual_288 0.714048 0.234169   0.185408 0.317731    1.269340    higher
            abs_slope_288 0.683798 0.233726   0.239222 0.228476    1.170947    higher
            temp_range_72 0.595310 0.221244   0.418764 0.150335    0.920033    higher
            temp_range_24 0.554965 0.220135   0.400724 0.151748    0.997261    higher
            temp_range_12 0.544549 0.218738   0.389954 0.152000    1.078738    higher
            temp_range_48 0.574851 0.218551   0.446925

f1 큰것: flatline_length  / is same temp

auc는 높음 abs_temp_change 72

In [ ]:
#sation별 성능

# ============================================================
# Station별 Feature AUC / Best F1
# ============================================================

station_results = []

for station in train_feat['station'].unique():

    sub = train_feat[
        train_feat['station'] == station
    ]

    y_station = sub['label'].astype(int)

    for feature in feature_cols:

        data = sub[
            [feature, 'label']
        ].dropna()

        if len(data) == 0:
            continue

        if data['label'].nunique() < 2:
            continue

        if data[feature].nunique() <= 1:
            continue

        x = data[feature].values
        y_true = data['label'].astype(int).values

        auc_raw = roc_auc_score(
            y_true,
            x
        )

        if auc_raw >= 0.5:
            auc = auc_raw
            score = x
            direction = 'higher'
        else:
            auc = 1 - auc_raw
            score = -x
            direction = 'lower'

        precision, recall, thresholds = (
            precision_recall_curve(
                y_true,
                score
            )
        )

        if len(thresholds) > 0:

            f1_scores = (
                2 * precision[:-1] * recall[:-1] /
                (
                    precision[:-1] +
                    recall[:-1] +
                    1e-12
                )
            )

            best_f1 = np.max(f1_scores)

        else:
            best_f1 = np.nan

        station_results.append({
            'station': station,
            'feature': feature,
            'AUC': auc,
            'Best_F1': best_f1,
            'direction': direction
        })


station_results = pd.DataFrame(
    station_results
)

In [ ]:
for station in station_results['station'].unique():

    print(
        f"\n========== {station} =========="
    )

    print(
        station_results[
            station_results['station'] == station
        ]
        .sort_values(
            'Best_F1',
            ascending=False
        )
        .head(15)
        .to_string(index=False)
    )


========== G-ORS ==========
station         feature      AUC  Best_F1 direction
  G-ORS flatline_length 0.720092 0.607958    higher
  G-ORS    is_same_temp 0.719611 0.590966    higher
  G-ORS   temp_range_12 0.568950 0.398654    higher
  G-ORS    temp_range_6 0.551475 0.396135    higher
  G-ORS      temp_std_6 0.551374 0.387975    higher
  G-ORS     temp_std_12 0.568359 0.373733    higher
  G-ORS      temp_std_3 0.536126 0.368816    higher
  G-ORS    temp_range_3 0.536111 0.366812    higher
  G-ORS     past_std_12 0.566503 0.366437    higher
  G-ORS   temp_range_24 0.597408 0.353020    higher
  G-ORS   temp_diff_abs 0.519093 0.343413    higher
  G-ORS abs_temp_diff_1 0.519093 0.343413    higher
  G-ORS     temp_std_24 0.595479 0.326590    higher
  G-ORS     temp_std_48 0.634200 0.322603    higher
  G-ORS     past_std_24 0.593946 0.320809    higher

========== I-ORS ==========
station                feature      AUC  Best_F1 direction
  I-ORS             temp_std_3 0.609551 0.371842   

In [ ]:
#station x layer별 feature separation

#    AUC / Best F1
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve
)


# ============================================================
# 1. Feature 목록
# ============================================================

drop_cols = [
    'station',
    'year',
    'layer',
    'time',
    'label',
    'anomaly_type',
    'dataset'
]

feature_cols = [
    col for col in train_feat.columns
    if col not in drop_cols
]


# ============================================================
# 2. Station × Layer별 계산
# ============================================================

results = []


for station in train_feat['station'].dropna().unique():

    for layer in sorted(
        train_feat.loc[
            train_feat['station'] == station,
            'layer'
        ].dropna().unique()
    ):

        sub = train_feat[
            (train_feat['station'] == station) &
            (train_feat['layer'] == layer)
        ].copy()

        for feature in feature_cols:

            data = sub[
                [feature, 'label']
            ].dropna()

            # 데이터 부족
            if len(data) < 20:
                continue

            # label이 한 종류뿐이면 계산 불가능
            if data['label'].nunique() < 2:
                continue

            # feature가 일정하면 계산 불가능
            if data[feature].nunique() <= 1:
                continue

            x = data[feature].values
            y = data['label'].astype(int).values

            # ------------------------------------------------
            # AUC
            # ------------------------------------------------

            auc_raw = roc_auc_score(y, x)

            # 어느 방향으로 anomaly가 나타나는지 확인
            if auc_raw >= 0.5:
                auc = auc_raw
                direction = 'higher'
                score = x
            else:
                auc = 1 - auc_raw
                direction = 'lower'
                score = -x

            # ------------------------------------------------
            # Best F1
            # ------------------------------------------------

            precision, recall, thresholds = (
                precision_recall_curve(
                    y,
                    score
                )
            )

            if len(thresholds) > 0:

                f1_scores = (
                    2 * precision[:-1] * recall[:-1]
                    /
                    (
                        precision[:-1]
                        + recall[:-1]
                        + 1e-12
                    )
                )

                best_idx = np.argmax(f1_scores)

                best_f1 = f1_scores[best_idx]
                best_threshold = thresholds[best_idx]

            else:

                best_f1 = np.nan
                best_threshold = np.nan

            # ------------------------------------------------
            # 저장
            # ------------------------------------------------

            results.append({
                'station': station,
                'layer': layer,
                'feature': feature,
                'AUC': auc,
                'Best_F1': best_f1,
                'direction': direction,
                'threshold': best_threshold,
                'N': len(data),
                'N_anomaly': y.sum()
            })


# ============================================================
# 3. DataFrame
# ============================================================

station_layer_results = pd.DataFrame(results)

print(
    station_layer_results.head()
)

  station  layer    feature       AUC   Best_F1 direction  threshold      N  \
0   G-ORS      1       temp  0.741690  0.192936    higher    21.5150  26503   
1   G-ORS      1       psal  0.736677  0.159838     lower   -28.3368  26499   
2   G-ORS      1      depth  0.628060  0.106811    higher     8.5900  26502   
3   G-ORS      1      month  0.703474  0.187030    higher     7.0000  26503   
4   G-ORS      1  dayofyear  0.691630  0.190704    higher   184.0000  26503   

   N_anomaly  
0       1067  
1       1067  
2       1067  
3       1067  
4       1067  


In [ ]:
# ============================================================
# ② Station × Layer별 Top 10 Feature
# ============================================================

for station in station_layer_results['station'].unique():

    for layer in sorted(
        station_layer_results.loc[
            station_layer_results['station'] == station,
            'layer'
        ].unique()
    ):

        result = station_layer_results[
            (station_layer_results['station'] == station) &
            (station_layer_results['layer'] == layer)
        ].sort_values(
            'AUC',
            ascending=False
        ).head(10)

        print(
            f"\n========== {station} / Layer {layer} =========="
        )

        print(
            result[
                [
                    'feature',
                    'AUC',
                    'Best_F1',
                    'direction',
                    'N_anomaly'
                ]
            ].to_string(index=False)
        )


========== G-ORS / Layer 1 ==========
                feature      AUC  Best_F1 direction  N_anomaly
         temp_range_288 0.807036 0.247406    higher       1067
              month_cos 0.799201 0.243987     lower       1067
           temp_std_288 0.796243 0.267445    higher       1067
           past_std_288 0.795401 0.267445    higher       1067
           temp_std_576 0.788420 0.253616    higher       1067
           past_std_576 0.787599 0.253616    higher       1067
         temp_range_576 0.768860 0.197618    higher       1067
  past_abs_residual_288 0.768325 0.264953    higher       1067
past_abs_dev_median_288 0.746743 0.270872    higher       1067
         temp_range_144 0.745367 0.254906    higher       1067

========== I-ORS / Layer 1 ==========
                feature      AUC  Best_F1 direction  N_anomaly
           temp_std_576 0.804771 0.217315    higher       1631
           past_std_576 0.803926 0.217139    higher       1631
  past_abs_residual_576 0.769991 0.24353

In [ ]:
# ============================================================
# ③ Station × Layer별 상위10개 Feature
# ============================================================

head_features = (
    station_layer_results
    .sort_values(
        ['station', 'layer', 'AUC'],
        ascending=[True, True, False]
    )
    .groupby(
        ['station', 'layer'],
        as_index=False
    )
    .head(3)
)

with pd.option_context(
    'display.max_rows', None,
    'display.max_columns', None,
    'display.width', 1000
):
    print(
        head_features[
            [
                'station',
                'layer',
                'feature',
                'AUC',
                'Best_F1',
                'direction',
                'N_anomaly'
            ]
        ].to_string(index=False)
    )

station  layer                   feature      AUC  Best_F1 direction  N_anomaly
  G-ORS      1            temp_range_288 0.807036 0.247406    higher       1067
  G-ORS      1                 month_cos 0.799201 0.243987     lower       1067
  G-ORS      1              temp_std_288 0.796243 0.267445    higher       1067
  I-ORS      1              temp_std_576 0.804771 0.217315    higher       1631
  I-ORS      1              past_std_576 0.803926 0.217139    higher       1631
  I-ORS      1     past_abs_residual_576 0.769991 0.243533    higher       1631
  I-ORS      2              temp_std_288 0.778022 0.222352    higher       1072
  I-ORS      2              past_std_288 0.776421 0.221574    higher       1072
  I-ORS      2            temp_range_288 0.764049 0.198933    higher       1072
  I-ORS      3     past_abs_residual_576 0.840953 0.542525    higher       1042
  I-ORS      3     past_abs_residual_288 0.838313 0.420349    higher       1042
  I-ORS      3   past_abs_dev_median_576

In [ ]:
#month_cos 이 너무 높음

# G-ORS L1 anomaly의 월별 비율
sub = train_feat[
    (train_feat['station'] == 'G-ORS') &
    (train_feat['layer'] == 1)
]

print(
    pd.crosstab(
        sub['month'],
        sub['label'],
        normalize='index'
    )
)

label       0.0       1.0
month                    
1      0.999726  0.000274
2      1.000000  0.000000
3      1.000000  0.000000
4      0.971093  0.028907
5      1.000000  0.000000
6      0.999336  0.000664
7      0.732452  0.267548
8      0.978142  0.021858
9      1.000000  0.000000
11     1.000000  0.000000


l1에 train 아노말리가 7월에 비정상적으로 집중되어있어서 month_Cos가 높은 분리력을 보임


csv 피처 저장

In [ ]:

# ============================================================
# Feature CSV 저장
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. 원본 train/test 복사
# ------------------------------------------------------------

train_features = make_features(train)
test_features = make_features(test)


# ------------------------------------------------------------
# 2. time 정렬
# ------------------------------------------------------------

train_features = train_features.sort_values(
    ['station', 'layer', 'time']
).reset_index(drop=True)

test_features = test_features.sort_values(
    ['station', 'layer', 'time']
).reset_index(drop=True)


# ------------------------------------------------------------
# 3. depth_diff 제거
# ------------------------------------------------------------

drop_features = [
    'depth_diff'
    'depth'
]

train_features = train_features.drop(
    columns=drop_features,
    errors='ignore'
)

test_features = test_features.drop(
    columns=drop_features,
    errors='ignore'
)


# ------------------------------------------------------------
# 4. 저장
# ------------------------------------------------------------

train_feature_file = '../train_features.csv'
test_feature_file = '../test_features.csv'

train_features.to_csv(
    train_feature_file,
    index=False,
    encoding='utf-8-sig'
)

test_features.to_csv(
    test_feature_file,
    index=False,
    encoding='utf-8-sig'
)


# ------------------------------------------------------------
# 5. 저장 확인
# ------------------------------------------------------------

print("Train 저장 완료")
print(train_feature_file)
print(train_features.shape)

print("\nTest 저장 완료")
print(test_feature_file)
print(test_features.shape)

print("\nTrain columns:")
print(train_features.columns.tolist())

print("\nTest columns:")
print(test_features.columns.tolist())



C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_23036\3466112505.py:381: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'past_residual_{w}'] = (
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_23036\3466112505.py:390: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'past_dev_median_{w}'] = (
C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_23036\3466112505.py:399: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfor

Train 저장 완료
../DATA/train_features.csv
(776706, 158)

Test 저장 완료
../DATA/test_features.csv
(169011, 156)

Train columns:
['station', 'year', 'layer', 'time', 'temp', 'psal', 'depth', 'label', 'anomaly_type', 'dataset', 'month', 'dayofyear', 'hour', 'month_sin', 'month_cos', 'temp_diff_1', 'abs_temp_diff_1', 'temp_diff_2', 'abs_temp_diff_2', 'temp_diff_3', 'abs_temp_diff_3', 'temp_mean_3', 'temp_std_3', 'temp_range_3', 'temp_residual_3', 'temp_dev_median_3', 'temp_zscore_3', 'temp_mean_6', 'temp_std_6', 'temp_range_6', 'temp_residual_6', 'temp_dev_median_6', 'temp_zscore_6', 'temp_mean_12', 'temp_std_12', 'temp_range_12', 'temp_residual_12', 'temp_dev_median_12', 'temp_zscore_12', 'temp_mean_24', 'temp_std_24', 'temp_range_24', 'temp_residual_24', 'temp_dev_median_24', 'temp_zscore_24', 'temp_mean_48', 'temp_std_48', 'temp_range_48', 'temp_residual_48', 'temp_dev_median_48', 'temp_zscore_48', 'temp_mean_72', 'temp_std_72', 'temp_range_72', 'temp_residual_72', 'temp_dev_median_72', 'temp

In [ ]:
# ============================================================
# 저장된 CSV 다시 읽어서 확인
# ============================================================

train_check = pd.read_csv(
    train_feature_file
)

test_check = pd.read_csv(
    test_feature_file
)

print("TRAIN")
print(train_check.shape)
print(train_check.head())

print("\nTEST")
print(test_check.shape)
print(test_check.head())

print("\nColumn 동일 여부")

train_cols = set(train_check.columns)
test_cols = set(test_check.columns)

print("Train에만 있는 컬럼:")
print(train_cols - test_cols)

print("\nTest에만 있는 컬럼:")
print(test_cols - train_cols)

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_23036\1963724135.py:5: DtypeWarning: Columns (0: anomaly_type) have mixed types. Specify dtype option on import or set low_memory=False.
  train_check = pd.read_csv(


TRAIN
(776706, 158)
  station  year  layer                       time     temp     psal  depth  \
0   G-ORS  2025      1  2025-01-01 00:00:00+09:00  14.4369  33.4490  8.434   
1   G-ORS  2025      1  2025-01-01 00:10:00+09:00  14.3951  33.4353  8.233   
2   G-ORS  2025      1  2025-01-01 00:20:00+09:00  14.3324  33.4002  8.347   
3   G-ORS  2025      1  2025-01-01 00:30:00+09:00  14.3886  33.4057  8.324   
4   G-ORS  2025      1  2025-01-01 00:40:00+09:00  14.3500  33.3691  8.272   

   label anomaly_type dataset  ...  past_residual_576  past_dev_median_576  \
0      0          NaN   train  ...                NaN                  NaN   
1      0          NaN   train  ...                NaN                  NaN   
2      0          NaN   train  ...                NaN                  NaN   
3      0          NaN   train  ...                NaN                  NaN   
4      0          NaN   train  ...                NaN                  NaN   

   past_zscore_576  past_abs_residual_576 